# Spleen Segmentation from CT — 2D U-Net

Trains a 2D U-Net (MONAI) to segment the spleen from abdominal CT (Medical Segmentation Decathlon, Task09), then evaluates per-case Dice on a held-out test set and produces the failure-analysis figures.

Run top to bottom on a GPU runtime (Runtime -> Change runtime type -> T4 GPU). All randomness is seeded for reproducibility. See README for results and the failure analysis.

In [ ]:
!pip install -q monai nibabel

In [ ]:
import os, random
from glob import glob
import numpy as np, torch
import torch.nn.functional as F
import nibabel as nib, pandas as pd
from monai.apps import download_and_extract

def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(42)

DATA_ROOT = "/content/spleen_project"; os.makedirs(DATA_ROOT, exist_ok=True)
data_dir = os.path.join(DATA_ROOT, "Task09_Spleen")
if not os.path.isdir(data_dir):
    download_and_extract(
        "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar",
        os.path.join(DATA_ROOT, "Task09_Spleen.tar"), DATA_ROOT)

def list_nii(folder):
    fs = sorted(glob(os.path.join(data_dir, folder, "*.nii.gz")))
    return [f for f in fs if not os.path.basename(f).startswith("._")]
data_dicts = [{"image": i, "label": l} for i, l in zip(list_nii("imagesTr"), list_nii("labelsTr"))]
random.seed(42); random.shuffle(data_dicts)
n = len(data_dicts); n_test = max(1, int(0.15*n)); n_val = max(1, int(0.15*n))
test_files, val_files, train_files = data_dicts[:n_test], data_dicts[n_test:n_test+n_val], data_dicts[n_test+n_val:]
print(f"train {len(train_files)} | val {len(val_files)} | test {len(test_files)}")

def apply_ct_window(x, level=50, width=350):
    low, high = level-width/2, level+width/2
    return (np.clip(x, low, high)-low)/(high-low)
def resize_2d(a, size, mode="bilinear"):
    t = a[None, None].float()
    if mode == "nearest": return F.interpolate(t, size=(size, size), mode="nearest")[0,0]
    return F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)[0,0]

class SpleenSlices2D(torch.utils.data.Dataset):
    def __init__(self, files, size=256, n_bg=0, augment=False):
        self.size, self.augment, self.items = size, augment, []
        for d in files:
            vol = apply_ct_window(nib.load(d["image"]).get_fdata()).astype(np.float32)
            gt  = (nib.load(d["label"]).get_fdata() > 0).astype(np.uint8)
            Z = vol.shape[2]
            spleen_z = [z for z in range(Z) if gt[:,:,z].sum() > 0]
            bg_z     = [z for z in range(Z) if gt[:,:,z].sum() == 0]
            chosen = list(spleen_z) + (random.sample(bg_z, min(n_bg, len(bg_z))) if n_bg and bg_z else [])
            for z in chosen: self.items.append((vol, gt, z))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        vol, gt, z = self.items[i]
        img = resize_2d(torch.from_numpy(np.ascontiguousarray(vol[:,:,z])), self.size, "bilinear")[None]
        msk = (resize_2d(torch.from_numpy(np.ascontiguousarray(gt[:,:,z])), self.size, "nearest") > 0.5).float()[None]
        if self.augment:
            if random.random()<0.5: img, msk = torch.flip(img,[2]), torch.flip(msk,[2])
            if random.random()<0.5: img, msk = torch.flip(img,[1]), torch.flip(msk,[1])
            k = random.randint(0,3)
            if k:                    img, msk = torch.rot90(img,k,[1,2]), torch.rot90(msk,k,[1,2])
            if random.random()<0.5:  img = torch.clamp(img*(0.8+0.4*random.random()), 0, 1)
        return img, msk

In [ ]:
from monai.networks.nets import UNet
from monai.losses import DiceLoss

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)

train_ds = SpleenSlices2D(train_files, n_bg=8, augment=True)
val_ds   = SpleenSlices2D(val_files,   n_bg=0, augment=False)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0)
val_dl   = torch.utils.data.DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
print("train slices:", len(train_ds), "| val slices:", len(val_ds))

model = UNet(spatial_dims=2, in_channels=1, out_channels=1,
             channels=(16,32,64,128), strides=(2,2,2), num_res_units=2).to(device)
loss_fn = DiceLoss(sigmoid=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 30
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

@torch.no_grad()
def val_dice():
    model.eval(); inter=den=0.0
    for x,y in val_dl:
        x,y = x.to(device), y.to(device)
        p = (torch.sigmoid(model(x))>0.5).float()
        inter += 2*(p*y).sum().item(); den += (p.sum()+y.sum()).item()
    return inter/den if den else 0.0

best, ckpt_name = 0.0, "best_model_day5.pth"
for ep in range(EPOCHS):
    model.train(); tot=0.0
    for x,y in train_dl:
        x,y = x.to(device), y.to(device)
        opt.zero_grad(); loss=loss_fn(model(x), y); loss.backward(); opt.step(); tot+=loss.item()
    sched.step(); d=val_dice()
    if d>best: best=d; torch.save(model.state_dict(), ckpt_name)
    print(f"epoch {ep+1:2d}/{EPOCHS} | loss {tot/len(train_dl):.4f} | val Dice {d:.4f} | best {best:.4f}")

print("best val Dice:", round(best,4))

In [ ]:
model.load_state_dict(torch.load(ckpt_name, map_location=device)); model.eval()

@torch.no_grad()
def evaluate_case(image_path, label_path, size=256, batch=16, threshold=0.5):
    vol = apply_ct_window(nib.load(image_path).get_fdata()).astype(np.float32)
    gt  = (nib.load(label_path).get_fdata() > 0).astype(np.float32)
    H,W,Z = vol.shape; pred_vol = np.zeros((H,W,Z), np.float32)
    for s in range(0,Z,batch):
        e=min(s+batch,Z)
        chunk=torch.stack([resize_2d(torch.from_numpy(np.ascontiguousarray(vol[:,:,z])), size) for z in range(s,e)])[:,None].to(device)
        binm=(torch.sigmoid(model(chunk))>threshold).float()
        back=F.interpolate(binm,size=(H,W),mode="nearest")[:,0].cpu().numpy()
        pred_vol[:,:,s:e]=np.transpose(back,(1,2,0))
    inter=float((pred_vol*gt).sum()); den=float(pred_vol.sum()+gt.sum())
    return (1.0 if den==0 else 2*inter/den), pred_vol, gt, vol

results, cache = [], {}
for d in test_files:
    name=os.path.basename(d["image"]).replace(".nii.gz","")
    dice,pred,gt,vol=evaluate_case(d["image"],d["label"]); tp=float((gt*pred).sum())
    results.append({"case":name,"dice":dice,"gt_voxels":int(gt.sum()),"pred_voxels":int(pred.sum()),
                    "precision":tp/pred.sum() if pred.sum() else 0.0,"recall":tp/gt.sum() if gt.sum() else 0.0})
    cache[name]=(vol,gt,pred); print(f"{name:>12s} | Dice {dice:.4f}")

df=pd.DataFrame(results).sort_values("gt_voxels").reset_index(drop=True)
df["excess_voxels"]=df.pred_voxels-df.gt_voxels; df["excess_pct"]=100*df.excess_voxels/df.gt_voxels
print(f"\nmean {df.dice.mean():.4f} | median {df.dice.median():.4f} | std {df.dice.std():.4f} | min {df.dice.min():.4f} | max {df.dice.max():.4f}")
df.to_csv("test_per_case_metrics.csv", index=False)
display(df.round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(df.gt_voxels, df.dice, s=80)
for _, r in df.iterrows(): ax.annotate(r.case,(r.gt_voxels,r.dice),fontsize=8,xytext=(5,5),textcoords="offset points")
ax.set_xlabel("spleen size (ground-truth voxels)"); ax.set_ylabel("Dice"); ax.set_ylim(0,1)
ax.set_title("Dice collapses as the spleen gets smaller"); plt.tight_layout()
plt.savefig("dice_vs_size.png", dpi=150); plt.show()

x,w = np.arange(len(df)), 0.35
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x-w/2, df.precision, w, label="precision"); ax.bar(x+w/2, df.recall, w, label="recall")
ax.set_xticks(x); ax.set_xticklabels(df.case, rotation=45, ha="right")
ax.set_ylim(0,1); ax.set_ylabel("score"); ax.legend()
ax.set_title("Recall stays high; precision falls on small spleens"); plt.tight_layout()
plt.savefig("precision_recall.png", dpi=150); plt.show()

def show_false_positives(name, k=3):
    vol, gt, pred = cache[name]
    bad = np.where((gt.sum((0,1))==0) & (pred.sum((0,1))>0))[0]
    if len(bad)==0: print("No background false positives for", name); return
    picks = bad[np.linspace(0,len(bad)-1,min(k,len(bad))).astype(int)]
    fig, ax = plt.subplots(1,len(picks),figsize=(4*len(picks),4)); ax=np.atleast_1d(ax)
    for j,z in enumerate(picks):
        ax[j].imshow(vol[:,:,z].T,cmap="gray",origin="lower")
        ax[j].imshow(np.ma.masked_where(pred[:,:,z].T==0,pred[:,:,z].T),cmap="autumn",alpha=0.6,origin="lower")
        ax[j].set_title(f"slice {z}: no spleen,\nmodel predicts (red)"); ax[j].axis("off")
    plt.suptitle(f"{name}: false positives on empty slices", y=1.03); plt.tight_layout()
    plt.savefig(f"false_positives_{name}.png", dpi=150); plt.show()

worst_two = df.sort_values("dice").head(2).case.tolist()
for name in worst_two: show_false_positives(name)